# 08 · Noticias y bolsa

**Objetivo:** medir si el sentimiento de las noticias de una empresa se relaciona con lo que hace su acción después. Los cuatro modelos del notebook 07 (M1 diccionario, M2 Transformer, M3 BERT y M4 FinBERT) puntúan los titulares de FNSPID que mencionan a su empresa, y se compara la rentabilidad de la acción, menos la del mercado, después de las noticias positivas, neutrales y negativas. Antes se comprueba con 300 titulares recientes etiquetados a mano si los modelos, entrenados con frases de PhraseBank, también saben leer titulares.

**Plan:**

| Paso | Qué se hace | Estado |
|---|---|---|
| 1 | Plantilla de 300 titulares de NewsAPI para etiquetar a mano, con la regla de etiquetado | hecho |
| 2 | Los cuatro modelos del 07: M1 desde su fichero; M2, M3 y M4 con el código compartido de `capas_keras.py` | hecho |
| 3 | Los cuatro modelos puntúan los titulares de FNSPID que mencionan a su empresa | hecho |
| 4 | Los cuatro modelos con los 300 titulares etiquetados a mano | hecho |
| 5 | SPY y rentabilidad de cada acción menos la del mercado | hecho |
| 6 | Sentimiento diario por empresa y estudio de eventos | hecho |

**Entradas:**
- `data/cache/noticias_newsapi.parquet` y `noticias_fnspid.parquet` (notebook 06)
- `data/cache/empresas_sec.parquet` (notebook 03): nombre de cada empresa
- `data/cache/precios_historicos.parquet` (notebook 02): precios diarios ajustados de las 1.000 empresas
- `etiquetas/titulares_a_mano.csv`: los 300 titulares, etiquetados a mano por el autor
- `data/modelos/` (notebook 07): `vocabulario.txt`, `m1_diccionario.json`, `m2_eleccion.json`, `m2_sin_pesos.keras`, `m3_mezclado_descongelado.keras` (M3, el corregido), `finbert_pretrain/` y `m4_finbert.weights.h5` (M4)
- `data/cache/predicciones_phrasebank.parquet` (notebook 07): para comprobar que los modelos se cargan bien

**Salidas:**
- `etiquetas/titulares_a_mano.csv`: la plantilla (solo si no existe, o se amplía por el final; nunca se borra una etiqueta)
- `data/cache/sentimiento_titulares.parquet`: el sentimiento de cada titular de FNSPID según cada modelo (una fila por titular y modelo)
- `data/cache/precios_spy.parquet`: precios diarios de SPY (yfinance), con el mismo ajuste y las mismas fechas que los del 02

In [1]:
import json
import re
import warnings
from pathlib import Path

from capas_keras import (   # va primero: fija el motor de Keras (TensorFlow)
    CAPAS_PROPIAS,
    SECUENCIA_FINBERT,
    cargar_m4,
    cargar_tokenizador_finbert,
    cargar_vectorizador,
)
import keras
import keras_hub
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
import yfinance as yf
from dotenv import load_dotenv

from funciones import (
    CARPETA_CACHE,
    CLASES_SENTIMIENTO,
    SEMILLA,
    clasificar_tono,
    comparar_f1,
    diferencia_medias_welch,
    evaluar_clasificacion,
    tono_loughran_mcdonald,
)

load_dotenv()
print("Keras", keras.__version__, "· motor:", keras.backend.backend())

Keras 3.15.1 · motor: tensorflow


## 1. ¿Saben los modelos leer titulares? La plantilla para etiquetar a mano

Los modelos del 07 se entrenaron y se examinaron con frases de **notas de prensa de empresas finlandesas de 2004 a 2010** (Financial PhraseBank). Aquí se aplican a **titulares de noticias de empresas de EEUU**, más cortos y con otra jerga. Si en este notebook el sentimiento no se relacionara con la bolsa, habría que saber si es porque no hay relación o porque los modelos no saben leer titulares. No existe un conjunto de titulares de nuestras empresas con su sentimiento etiquetado, así que el autor etiqueta **300 a mano**. Al principio se previeron 200; antes de etiquetar ninguno y de ver ningún resultado, el autor amplió a 300 para medir con más precisión (el intervalo del F1 pasa de unos ±0,065 a unos ±0,055).

**Qué titulares:** los recientes de NewsAPI (notebook 06), que ya mencionan a su empresa. Son de otra época que los de FNSPID, así que es una prueba **fuera de dominio** en dos sentidos: otro tipo de texto que PhraseBank y otras fechas. Se eligen al azar con la semilla del proyecto, sin titulares repetidos y **como mucho 5 por empresa**, para que unas pocas empresas muy citadas (Amazon tiene el 14 % de los titulares) no se lleven la muestra. Para que la muestra sea siempre la misma aunque se vuelva a ejecutar el 06, se usan solo los titulares descargados hasta el 16 nov 2026, el día en que se hizo.

**La regla de etiquetado** se escribió antes de empezar, en `etiquetas/regla_de_etiquetado.md`. Es la misma pregunta que se hizo a los anotadores de PhraseBank: desde el punto de vista de un inversor en esa empresa, ¿el titular es una noticia buena (`positive`), mala (`negative`) o neutra (`neutral`) para el precio de su acción? Solo con el titular, que es lo que ven los modelos; en la duda, `neutral`; y sin mirar lo que dicen los modelos.

**Titulares que no tratan de la empresa.** El filtro del 06 (`menciona_empresa`) acepta un titular si contiene el ticker o el nombre de la empresa, y a veces coinciden con una palabra corriente (*principal*, *hinge*, *holding*, *SNAP*). Esos titulares no se quitan: también están entre los de FNSPID, así que forman parte de lo que leerán los modelos. Se marcan en una columna, `trata_de_la_empresa` (`sí` o `no`), y se etiquetan como `neutral`, porque no deberían mover la acción. Así se pueden dar dos resultados: con los 300 titulares (lo que pasará en FNSPID) y solo con los que tratan de la empresa (cómo leen los modelos un titular de verdad). Y se mide de paso cuántas veces se equivoca el filtro.

In [2]:
CARPETA_ETIQUETAS = Path("etiquetas")
RUTA_ETIQUETADO = CARPETA_ETIQUETAS / "titulares_a_mano.csv"
TITULARES_PREVISTOS = 200   # la primera plantilla
TITULARES_A_MANO = 300      # ampliada antes de etiquetar: los 100 nuevos van detrás, con id 201-300
MAXIMO_POR_EMPRESA = 5
FECHA_MUESTRA = pd.Timestamp("2026-11-16")   # titulares descargados hasta el día en que se hizo la muestra

noticias_recientes = pd.read_parquet(CARPETA_CACHE / "noticias_newsapi.parquet")
noticias_recientes = noticias_recientes[noticias_recientes["descargado"] <= FECHA_MUESTRA]
print(f"Titulares de NewsAPI: {len(noticias_recientes):,} · distintos: {noticias_recientes['titular'].nunique():,}"
      f" · empresas: {noticias_recientes['ticker'].nunique()}"
      f" · del {noticias_recientes['fecha'].min():%d %b %Y} al {noticias_recientes['fecha'].max():%d %b %Y}")

Titulares de NewsAPI: 3,036 · distintos: 2,671 · empresas: 106 · del 20 Sep 2026 al 15 Nov 2026


La muestra: se barajan los titulares distintos con la semilla, se quedan los 5 primeros de cada empresa y, de ellos, se eligen 200 al azar. Los 100 de la ampliación se eligen igual entre los candidatos que quedan, así que los 200 primeros no cambian. Se añade el nombre de la empresa para saber de cuál se juzga cada titular.


In [3]:
nombres = pd.read_parquet(CARPETA_CACHE / "empresas_sec.parquet").set_index("ticker")["name"]

candidatos = (noticias_recientes.drop_duplicates("titular")
              .sample(frac=1, random_state=SEMILLA)          # orden al azar, con semilla
              .groupby("ticker").head(MAXIMO_POR_EMPRESA))   # como mucho 5 por empresa
primeros = candidatos.sample(TITULARES_PREVISTOS, random_state=SEMILLA)
ampliacion = candidatos.drop(primeros.index).sample(TITULARES_A_MANO - TITULARES_PREVISTOS, random_state=SEMILLA)
muestra = pd.concat([primeros, ampliacion])

plantilla = pd.DataFrame({
    "id": range(1, TITULARES_A_MANO + 1),
    "ticker": muestra["ticker"].to_numpy(),
    "empresa": muestra["ticker"].map(nombres).to_numpy(),
    "titular": muestra["titular"].to_numpy(),
    "trata_de_la_empresa": "",
    "etiqueta": "",
})
print(f"Candidatos (como mucho {MAXIMO_POR_EMPRESA} por empresa): {len(candidatos)} · elegidos: {len(plantilla)}"
      f" · empresas en la muestra: {plantilla['ticker'].nunique()}")
plantilla.head()

Candidatos (como mucho 5 por empresa): 457 · elegidos: 300 · empresas en la muestra: 105


,id,ticker,empresa,titular,trata_de_la_empresa,etiqueta
0,1,KMI,"KINDER MORGAN, INC.",Energy Transfer vs. Kinder Morgan: Which Natur...,,
1,2,PFG,PRINCIPAL FINANCIAL GROUP INC,School principal documents Indonesia's worst w...,,
2,3,DDOG,"Datadog, Inc.",Here Are Monday’s Top Wall Street Analyst Rese...,,
3,4,TMUS,"T-Mobile US, Inc.",Do You Like Cash? Consider T-Mobile Stock,,
4,5,PENG,"Penguin Solutions, Inc.",Penguin Solutions Trumpets AI-Driven Beat-And-...,,


La plantilla se escribe **solo si no existe**. Si ya existe con menos titulares (la de 200), se le añaden por el final los que faltan, conservando lo ya etiquetado. Volver a ejecutar el notebook nunca borra una etiqueta.


In [4]:
if not RUTA_ETIQUETADO.exists():
    CARPETA_ETIQUETAS.mkdir(exist_ok=True)
    plantilla.to_csv(RUTA_ETIQUETADO, index=False, encoding="utf-8")
    print(f"Plantilla escrita: {RUTA_ETIQUETADO} ({len(plantilla)} titulares)")
else:
    actual = pd.read_csv(RUTA_ETIQUETADO, sep=None, engine="python", keep_default_na=False)
    if len(actual) < len(plantilla):
        # Las filas que ya hay tienen que ser las primeras de la plantilla: solo se añaden las que faltan
        assert actual[["id", "ticker", "titular"]].equals(plantilla[["id", "ticker", "titular"]].head(len(actual)))
        pd.concat([actual, plantilla.iloc[len(actual):]], ignore_index=True).to_csv(RUTA_ETIQUETADO, index=False, encoding="utf-8")
        print(f"Plantilla ampliada: de {len(actual)} a {len(plantilla)} titulares, sin tocar los que ya había")
    else:
        print(f"{RUTA_ETIQUETADO} ya existe con {len(actual)} titulares: no se cambia")


etiquetas/titulares_a_mano.csv ya existe con 300 titulares: no se cambia


Cuántos titulares hay etiquetados. El fichero se lee de forma flexible (con comas o con punto y coma, que es como lo guardan algunas hojas de cálculo) y sin tener en cuenta mayúsculas ni espacios (*si* vale como *sí*). Avisa de las filas a medias o mal escritas y de las que no tratan de la empresa pero no son `neutral`.

In [5]:
etiquetado = pd.read_csv(RUTA_ETIQUETADO, sep=None, engine="python", keep_default_na=False)
etiquetado["etiqueta"] = etiquetado["etiqueta"].str.strip().str.lower()
etiquetado["trata_de_la_empresa"] = etiquetado["trata_de_la_empresa"].str.strip().str.lower().replace({"si": "sí"})
hechos = etiquetado["etiqueta"].isin(CLASES_SENTIMIENTO) & etiquetado["trata_de_la_empresa"].isin(["sí", "no"])
empezados = (etiquetado["etiqueta"] != "") | (etiquetado["trata_de_la_empresa"] != "")
mal_escritos = etiquetado[empezados & ~hechos]
no_neutrales_ajenos = etiquetado[(etiquetado["trata_de_la_empresa"] == "no") & (etiquetado["etiqueta"] != "neutral")]

print(f"Etiquetados: {hechos.sum()} de {len(etiquetado)}")
print("Reparto:", etiquetado.loc[hechos, "etiqueta"].value_counts().to_dict(),
      "· no tratan de la empresa:", (etiquetado.loc[hechos, "trata_de_la_empresa"] == "no").sum())
if len(mal_escritos):
    print("A medias o mal escritos (etiqueta: positive, negative o neutral; trata_de_la_empresa: sí o no):")
    display(mal_escritos[["id", "titular", "trata_de_la_empresa", "etiqueta"]])
if len(no_neutrales_ajenos):
    print("No tratan de la empresa, pero su etiqueta no es neutral:")
    display(no_neutrales_ajenos[["id", "titular", "trata_de_la_empresa", "etiqueta"]])


Etiquetados: 300 de 300
Reparto: {'neutral': 173, 'positive': 72, 'negative': 55} · no tratan de la empresa: 29


## 2. Los cuatro modelos del notebook 07

Se cargan tal como quedaron en el 07, sin volver a entrenar nada:

- **M1, diccionario:** las dos listas de palabras y el umbral de "neutral" elegido en la validación (`m1_diccionario.json`).
- **M2, Transformer:** la versión elegida en la validación, sin pesos de clase (`m2_eleccion.json`).
- **M3, BERT:** el **corregido** (secciones 6 y 7 del 07), que es el oficial.
- **M4, FinBERT:** FinBERT con los pesos ajustados en la sección 8 del 07.

Para cargar M2 y M3, Keras necesita las clases de sus capas propias (`TransformerEncoder` y `PositionalEmbedding`) y la misma estandarización del texto; M4 se construye igual que en el 07 (FinBERT con la GELU exacta y la cabeza `[CLS]`) y se le cargan los pesos ajustados. Todo está en `capas_keras.py`, el mismo código que usa el 07, en un archivo aparte de `funciones.py` porque necesita Keras, que los demás notebooks no usan.

**Comprobación de que se cargan bien:** los cuatro modelos tienen que repetir exactamente lo que dijeron en el 07 sobre las 967 frases de la validación de PhraseBank.

In [6]:
CARPETA_MODELOS = CARPETA_CACHE.parent / "modelos"
SENTIMIENTO_NUMERO = {"negative": -1, "neutral": 0, "positive": 1}

# Keras avisa de que TransformerEncoder "destruye la máscara"; es lo mismo que en el 07 y en los apuntes
warnings.filterwarnings("ignore", message=".*does not support masking.*")
# y, al cargar, de que sus capas no tienen método build(): tampoco lo tienen en los apuntes, y los pesos se cargan bien
# (lo comprueba la celda siguiente, que repite exactamente las predicciones del 07)
warnings.filterwarnings("ignore", message=".*does not have a `build\\(\\)` method.*")
# y, al cargar M4, de que no carga el estado del optimizador guardado en el 07: aquí solo se predice
warnings.filterwarnings("ignore", message=".*Skipping variable loading for optimizer.*")

m1 = json.loads((CARPETA_MODELOS / "m1_diccionario.json").read_text(encoding="utf-8"))
palabras_negativas, palabras_positivas = set(m1["palabras_negativas"]), set(m1["palabras_positivas"])

vectorize_layer = cargar_vectorizador(CARPETA_MODELOS / "vocabulario.txt")
version_m2 = json.loads((CARPETA_MODELOS / "m2_eleccion.json").read_text(encoding="utf-8"))["version"]
fichero_m2 = {"sin pesos de clase": "m2_sin_pesos.keras", "con pesos de clase": "m2_con_pesos.keras"}[version_m2]
m2 = keras.models.load_model(CARPETA_MODELOS / fichero_m2, custom_objects=CAPAS_PROPIAS, compile=False)
m3 = keras.models.load_model(CARPETA_MODELOS / "m3_mezclado_descongelado.keras", custom_objects=CAPAS_PROPIAS, compile=False)
m4 = cargar_m4(CARPETA_MODELOS / "finbert_pretrain", CARPETA_MODELOS / "m4_finbert.weights.h5")
tokenizador_finbert = cargar_tokenizador_finbert(CARPETA_MODELOS / "finbert_pretrain")
print(f"M1: umbral {m1['umbral']}, {len(palabras_negativas):,} palabras negativas y {len(palabras_positivas):,} positivas"
      f" · M2: {version_m2} ({fichero_m2}) · M3: el corregido · M4: FinBERT ajustado")

M1: umbral 0.0, 2,345 palabras negativas y 347 positivas · M2: sin pesos de clase (m2_sin_pesos.keras) · M3: el corregido · M4: FinBERT ajustado


Cada modelo se usa con una función que recibe una lista de textos y devuelve, para cada uno, la **clase** y la **confianza**:

- **M2, M3 y M4:** la clase con más probabilidad, y esa probabilidad como confianza (entre 1/3 y 1).
- **M1:** la clase según el tono y el umbral, como en el 07. Como confianza, la **proporción de palabras del texto que están en el diccionario**, positivas o negativas. Un titular sin ninguna tiene confianza 0: M1 lo llama neutral porque no sabe nada de él.

**M4 se puntúa por grupos de longitud.** En el 07 cada frase se rellenaba hasta 136 tokens, pero la mayoría de los titulares ocupa menos de 32, y rellenarlos todos hasta 136 multiplicaría el tiempo por cuatro (unas dos horas en lugar de unos 20 minutos). Con la cabeza `[CLS]`, el resultado no depende del relleno (se comprueba en el 07), así que cada texto se rellena solo hasta el primer tope que le cabe: 32, 64 o 136 tokens.

In [7]:
def puntuar_diccionario(textos: list[str]) -> pd.DataFrame:
    """Clase de M1 para cada texto y su confianza: la proporción de sus palabras que están en el diccionario."""
    palabras_con_tono = palabras_negativas | palabras_positivas
    clases, confianzas = [], []
    for texto in textos:
        clases.append(clasificar_tono(tono_loughran_mcdonald(texto, palabras_negativas, palabras_positivas), m1["umbral"]))
        palabras = re.findall(r"[A-Z]+", texto.upper())   # las mismas palabras que cuenta tono_loughran_mcdonald
        confianzas.append(sum(palabra in palabras_con_tono for palabra in palabras) / len(palabras) if palabras else 0.0)
    return pd.DataFrame({"clase": clases, "confianza": confianzas})


def puntuar_red(modelo, textos: list[str]) -> pd.DataFrame:
    """Clase con más probabilidad para cada texto (M2 o M3) y esa probabilidad como confianza."""
    probabilidades = modelo.predict(vectorize_layer(textos).numpy(), batch_size=256, verbose=0)
    return pd.DataFrame({"clase": [CLASES_SENTIMIENTO[numero] for numero in probabilidades.argmax(axis=1)],
                         "confianza": probabilidades.max(axis=1)})


GRUPOS_DE_LONGITUD = [32, 64, SECUENCIA_FINBERT]   # tokens, con [CLS] y [SEP]


def puntuar_finbert(textos: list[str]) -> pd.DataFrame:
    """Clase de M4 para cada texto y su probabilidad, rellenando cada texto solo hasta el primer tope que le cabe."""
    longitudes = np.array([len(tokens) + 2 for tokens in tokenizador_finbert(textos)])   # + 2: [CLS] y [SEP]
    if longitudes.max() > GRUPOS_DE_LONGITUD[-1]:
        raise ValueError(f"Un texto ocupa {longitudes.max()} tokens, más de los {GRUPOS_DE_LONGITUD[-1]} con los que se ajustó M4")
    probabilidades = np.zeros((len(textos), 3))
    desde = 0
    for hasta in GRUPOS_DE_LONGITUD:
        posiciones = np.where((longitudes > desde) & (longitudes <= hasta))[0]
        if len(posiciones) > 0:
            preprocesador = keras_hub.models.BertTextClassifierPreprocessor(tokenizador_finbert, sequence_length=hasta)
            probabilidades[posiciones] = m4.predict(preprocesador([textos[i] for i in posiciones]), batch_size=128, verbose=0)
        desde = hasta
    return pd.DataFrame({"clase": [CLASES_SENTIMIENTO[numero] for numero in probabilidades.argmax(axis=1)],
                         "confianza": probabilidades.max(axis=1)})


MODELOS = {
    "M1 diccionario": puntuar_diccionario,
    "M2 Transformer": lambda textos: puntuar_red(m2, textos),
    "M3 BERT": lambda textos: puntuar_red(m3, textos),
    "M4 FinBERT": puntuar_finbert,
}

In [8]:
predicciones_07 = pd.read_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet")
validacion_07 = predicciones_07[predicciones_07["parte"] == "validacion"]

coinciden_con_el_07 = {
    nombre: (puntuar(validacion_07["frase"].tolist())["clase"].to_numpy() == validacion_07[nombre].to_numpy()).mean()
    for nombre, puntuar in MODELOS.items()
}
print(f"Frases de la validación de PhraseBank: {len(validacion_07)}")
for nombre, proporcion in coinciden_con_el_07.items():
    print(f"  {nombre}: dice lo mismo que en el 07 en el {proporcion:.1%}")

Frases de la validación de PhraseBank: 967
  M1 diccionario: dice lo mismo que en el 07 en el 100.0%
  M2 Transformer: dice lo mismo que en el 07 en el 100.0%
  M3 BERT: dice lo mismo que en el 07 en el 100.0%
  M4 FinBERT: dice lo mismo que en el 07 en el 100.0%


**Conclusión:** los cuatro modelos se cargan sin volver a entrenar y repiten **exactamente** lo que dijeron en el 07 (100 % de las frases de la validación); en M4, también puntuando por grupos de longitud. El código compartido de `capas_keras.py` carga los modelos tal como quedaron.

## 3. ¿Qué sentimiento tienen los titulares de FNSPID?

Los modelos puntúan los **192.069 titulares de FNSPID que mencionan a su empresa** (octubre 2016 – junio 2020, notebook 06). Los demás titulares de FNSPID son noticias en las que la empresa no aparece (listas de valores, noticias del mercado en general) y no cuentan para su sentimiento.

Algunos titulares mencionan a dos empresas y están dos veces, una con cada una. Cada texto distinto se puntúa una sola vez y el resultado se copia a sus filas.

**Salida:** una tabla con una fila por titular y modelo: empresa (`ticker`), fecha, titular, modelo, sentimiento (−1 negativo, 0 neutral, 1 positivo) y confianza. Se guarda en `data/cache/sentimiento_titulares.parquet`, y la caché es por modelo: si un modelo ya está en el fichero, se lee de disco, y solo se puntúa con los que falten (M4 se añadió después que M1–M3). M1–M3 tardan unos minutos en total y M4 unos 20.

In [9]:
RUTA_SENTIMIENTO = CARPETA_CACHE / "sentimiento_titulares.parquet"

noticias_fnspid = pd.read_parquet(CARPETA_CACHE / "noticias_fnspid.parquet")
titulares = noticias_fnspid.loc[noticias_fnspid["sobre_la_empresa"], ["ticker", "fecha", "titular"]]
textos = titulares["titular"].drop_duplicates().tolist()
print(f"Titulares que mencionan a su empresa: {len(titulares):,} · textos distintos: {len(textos):,}"
      f" · empresas: {titulares['ticker'].nunique()}")

partes = [pd.read_parquet(RUTA_SENTIMIENTO)] if RUTA_SENTIMIENTO.exists() else []
ya_puntuados = set(partes[0]["modelo"]) if partes else set()
print("Leídos de disco:", sorted(ya_puntuados) or "ninguno")
for nombre, puntuar in MODELOS.items():
    if nombre not in ya_puntuados:
        puntos = puntuar(textos).assign(titular=textos, modelo=nombre)
        puntos["sentimiento"] = puntos["clase"].map(SENTIMIENTO_NUMERO)
        partes.append(titulares.merge(puntos, on="titular")[["ticker", "fecha", "titular", "modelo", "sentimiento", "confianza"]])
        print(f"  {nombre}: puntuado")
sentimiento = pd.concat(partes, ignore_index=True)
if set(sentimiento["modelo"]) != ya_puntuados:
    sentimiento.to_parquet(RUTA_SENTIMIENTO, index=False)
    print("Guardado con los modelos nuevos")
sentimiento.head()

Titulares que mencionan a su empresa: 192,069 · textos distintos: 183,485 · empresas: 633
Leídos de disco: ['M1 diccionario', 'M2 Transformer', 'M3 BERT', 'M4 FinBERT']


,ticker,fecha,titular,modelo,sentimiento,confianza
0,A,2016-11-10,"Leerink Swann Assumes Agilent Technologies, In...",M1 diccionario,1,0.1
1,A,2016-11-15,Agilent Technologies Sees FY17 Rev. $4.35B-$4....,M1 diccionario,0,0.0
2,A,2016-11-15,Agilent Technologies Reports Q4 EPS $0.59 vs. ...,M1 diccionario,0,0.0
3,A,2016-11-15,Agilent Technologies' (A) CEO Mike McMullen on...,M1 diccionario,0,0.0
4,A,2016-11-15,Agilent Technologies Inc. 2016 Q4 - Results - ...,M1 diccionario,0,0.0


**¿Cuántos titulares ve cada modelo como positivos, neutrales y negativos, y con qué confianza?** Como referencia, en PhraseBank el 59 % de las frases es neutral, el 28 % positiva y el 12,5 % negativa.

In [10]:
reparto = (pd.crosstab(sentimiento["modelo"], sentimiento["sentimiento"], normalize="index")
           .rename(columns={-1: "negativo", 0: "neutral", 1: "positivo"}) * 100)
reparto["confianza mediana"] = sentimiento.groupby("modelo")["confianza"].median()
reparto.round(1)

sentimiento,negativo,neutral,positivo,confianza mediana
modelo,,,,
M1 diccionario,13.8,74.6,11.6,0.0
M2 Transformer,7.9,67.2,24.9,0.8
M3 BERT,5.1,52.6,42.3,0.7
M4 FinBERT,14.5,57.1,28.5,0.8


**¿Coinciden los modelos entre sí?** Proporción de titulares a los que cada par de modelos da el mismo sentimiento.

In [11]:
por_titular = sentimiento.pivot_table(index=["ticker", "fecha", "titular"], columns="modelo", values="sentimiento")
pares = [("M1 diccionario", "M2 Transformer"), ("M1 diccionario", "M3 BERT"), ("M2 Transformer", "M3 BERT"),
         ("M1 diccionario", "M4 FinBERT"), ("M2 Transformer", "M4 FinBERT"), ("M3 BERT", "M4 FinBERT")]
for a, b in pares:
    print(f"{a} y {b}: el mismo sentimiento en el {(por_titular[a] == por_titular[b]).mean():.1%} de los titulares")

M1 diccionario y M2 Transformer: el mismo sentimiento en el 60.0% de los titulares
M1 diccionario y M3 BERT: el mismo sentimiento en el 49.6% de los titulares
M2 Transformer y M3 BERT: el mismo sentimiento en el 58.8% de los titulares
M1 diccionario y M4 FinBERT: el mismo sentimiento en el 61.9% de los titulares
M2 Transformer y M4 FinBERT: el mismo sentimiento en el 60.2% de los titulares
M3 BERT y M4 FinBERT: el mismo sentimiento en el 61.6% de los titulares


Unos cuantos titulares al azar (con la semilla) con lo que dice cada modelo, para ver cómo leen los titulares:

In [12]:
with pd.option_context("display.max_colwidth", 110):
    display(por_titular.sample(12, random_state=SEMILLA).astype(int)
            .reset_index(["ticker", "fecha"], drop=True)
            .rename(columns={"M1 diccionario": "M1", "M2 Transformer": "M2", "M3 BERT": "M3", "M4 FinBERT": "M4"}))

modelo,M1,M2,M3,M4
titular,,,,
"Thinking about buying stock in Aurora Cannabis, American Express, PepsiCo, Wells Fargo, or Walmart?",0,0,0,0
Trimble +4.9% after $1.2B deal for Viewpoint,0,0,0,0
JPMorgan gets caught up in Venezuelan struggle,0,1,1,-1
Are Investor Concerns On Monster Beverages Misplaced?,-1,0,0,-1
Duke Energy launches 25M-share public offering,0,0,0,0
"Imperial Capital Maintains Outperform on Matador Resources, Raises Price Target to $25",1,1,1,1
FirstEnergy Invites Customers to Enter Holiday Lights Contest,0,0,0,0
"Evercore ISI Group Initiates Coverage On Incyte with In-Line Rating, Announces $135.00 Price Target",0,0,1,1
Why Is FactSet (FDS) Down 4.6% Since Last Earnings Report?,0,-1,0,-1


**Conclusión: los cuatro modelos leen los mismos titulares de forma muy distinta.** Solo coinciden los cuatro en el 32 % de los titulares, y cada par coincide entre el 50 % y el 62 %.

- **M1 se queda casi siempre en neutral (75 %).** El 74 % de los titulares no tiene ninguna palabra del diccionario: es el mismo problema de cobertura que en PhraseBank, y peor, porque un titular tiene pocas palabras.
- **M2 tiene un reparto parecido al de PhraseBank**, con más positivos (25 %) que negativos (8 %).
- **M3 ve muchos más positivos que el resto (42 %).** Es su debilidad con las negativas que ya se vio en el 07: de los titulares que M4 llama negativos, M3 da por positivos más de la mitad (7,9 % de 14,5 %).
- **M4, el mejor modelo en PhraseBank, es el que más negativos ve (14,5 %),** con un 57 % de neutrales y un 28 % de positivos. Es el único que lee como negativos tanto *Why Is FactSet Down 4.6%* como *JPMorgan gets caught up in Venezuelan struggle*.
- **Un límite de M2 y M3:** la estandarización de los apuntes borra los signos, así que *Trimble +4.9%* les llega como *trimble 49*. El tokenizador de FinBERT sí conserva `+` y `%`, pero M4 también lo da por neutral.

Qué modelo lee mejor los titulares no se puede saber mirando solo estos repartos: lo dice la sección 4, con los 300 titulares etiquetados a mano.

## 4. ¿Saben los modelos leer titulares? Los 300 etiquetados a mano

El examen fuera de dominio que se preparó en la sección 1: los 300 titulares recientes de NewsAPI, etiquetados a mano por el autor con su regla **antes de ver lo que decía ningún modelo de ellos**. Los modelos ya están fijados y aquí no se decide nada con estos titulares, así que se miden una sola vez, con las mismas funciones que el test de PhraseBank (`evaluar_clasificacion` y `comparar_f1`).

Se mide con dos conjuntos:

- **Los 300.**
- **Los 271 que tratan de su empresa.** Los 29 restantes son titulares que el filtro dejó pasar por error, todos `neutral` por la regla. Si un modelo acierta mucho en ellos por decir "neutral", el primer conjunto le favorece.

Las preguntas son las del 07: M2 − M1, M3 − M2 (la única comparación limpia del preentrenamiento), M4 − M2 y M4 − M3 (recetas completas). Con 300 titulares los intervalos son más anchos que con las 968 frases del test, así que habrá diferencias que no se puedan afirmar.

In [13]:
a_mano = etiquetado[["id", "ticker", "titular", "trata_de_la_empresa", "etiqueta"]].copy()
a_mano["siempre neutral"] = "neutral"
for nombre, puntuar in MODELOS.items():
    a_mano[nombre] = puntuar(a_mano["titular"].tolist())["clase"].to_numpy()

MODELOS_A_MANO = ["siempre neutral", *MODELOS]
CONJUNTOS_A_MANO = {"los 300": a_mano, "los 271 que tratan de su empresa": a_mano[a_mano["trata_de_la_empresa"] == "sí"]}
print({nombre: len(titulares_conjunto) for nombre, titulares_conjunto in CONJUNTOS_A_MANO.items()},
      "· etiquetas:", a_mano["etiqueta"].value_counts().to_dict())

resultados_a_mano, evaluaciones_a_mano = {}, {}
for nombre_conjunto, titulares_conjunto in CONJUNTOS_A_MANO.items():
    for modelo in MODELOS_A_MANO:
        e = evaluar_clasificacion(titulares_conjunto["etiqueta"], titulares_conjunto[modelo])
        evaluaciones_a_mano[(nombre_conjunto, modelo)] = e
        resultados_a_mano[(nombre_conjunto, modelo)] = {"F1 macro": e["f1_macro"], "IC 95 % inferior": e["f1_ic_inferior"],
                                                         "IC 95 % superior": e["f1_ic_superior"], "exactitud": e["exactitud"]}
pd.DataFrame(resultados_a_mano).T.round(3)

{'los 300': 300, 'los 271 que tratan de su empresa': 271} · etiquetas: {'neutral': 173, 'positive': 72, 'negative': 55}


F1 macro  IC 95 % inferior  \
los 300                          siempre neutral     0.244             0.229   
                                 M1 diccionario      0.460             0.403   
                                 M2 Transformer      0.376             0.312   
                                 M3 BERT             0.375             0.319   
                                 M4 FinBERT          0.626             0.555   
los 271 que tratan de su empresa siempre neutral     0.231             0.214   
                                 M1 diccionario      0.466             0.396   
                                 M2 Transformer      0.370             0.305   
                                 M3 BERT             0.368             0.315   
                                 M4 FinBERT          0.630             0.568   

                                                  IC 95 % superior  exactitud  
los 300                          siempre neutral             0.258      0.577  
                                 M1 diccionario              0.519      0.557  
                                 M2 Transformer              0.439      0.507  
                                 M3 BERT                     0.431      0.527  
                                 M4 FinBERT                  0.688      0.687  
los 271 que tratan de su empresa siempre neutral             0.247      0.531  
                                 M1 diccionario              0.532      0.546  
                                 M2 Transformer              0.435      0.483  
                                 M3 BERT                     0.425      0.498  
                                 M4 FinBERT                  0.690      0.675

**¿Cuánto se pierde al salir de PhraseBank?** F1 macro de cada modelo en el test de PhraseBank (notebook 07) y en los 300 titulares:

In [14]:
test_07 = predicciones_07[predicciones_07["parte"] == "test"]
pd.DataFrame({
    modelo: {"test de PhraseBank": evaluar_clasificacion(test_07["etiqueta"], test_07[modelo])["f1_macro"],
             "300 titulares a mano": resultados_a_mano[("los 300", modelo)]["F1 macro"]}
    for modelo in MODELOS_A_MANO
}).T.round(3)

,test de PhraseBank,300 titulares a mano
siempre neutral,0.248,0.244
M1 diccionario,0.502,0.460
M2 Transformer,0.673,0.376
M3 BERT,0.586,0.375
M4 FinBERT,0.813,0.626


Las diferencias entre modelos, con su intervalo (*bootstrap* emparejado sobre los mismos titulares):

In [15]:
PREGUNTAS = {
    "¿Aprender de ejemplos supera al diccionario? (M2 − M1)": ("M2 Transformer", "M1 diccionario"),
    "¿Aporta preentrenar? (M3 − M2), la única limpia": ("M3 BERT", "M2 Transformer"),
    "¿Partir de FinBERT supera a entrenar solo con etiquetas? (M4 − M2)": ("M4 FinBERT", "M2 Transformer"),
    "¿Partir de FinBERT supera a preentrenar uno propio? (M4 − M3)": ("M4 FinBERT", "M3 BERT"),
}

diferencias_a_mano = {}
for nombre_conjunto, titulares_conjunto in CONJUNTOS_A_MANO.items():
    for pregunta, (modelo_a, modelo_b) in PREGUNTAS.items():
        d = comparar_f1(titulares_conjunto["etiqueta"], titulares_conjunto[modelo_a], titulares_conjunto[modelo_b])
        diferencias_a_mano[(nombre_conjunto, pregunta)] = {
            "diferencia": round(d["diferencia"], 3), "IC 95 % inferior": round(d["ic_inferior"], 3),
            "IC 95 % superior": round(d["ic_superior"], 3),
            "¿concluyente?": "sí" if d["ic_inferior"] > 0 or d["ic_superior"] < 0 else "no (el 0 está dentro)"}
pd.DataFrame(diferencias_a_mano).T

diferencia  \
los 300                          ¿Aprender de ejemplos supera al diccionario? (M...     -0.085   
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia        -0.001   
                                 ¿Partir de FinBERT supera a entrenar solo con e...       0.25   
                                 ¿Partir de FinBERT supera a preentrenar uno pro...      0.251   
los 271 que tratan de su empresa ¿Aprender de ejemplos supera al diccionario? (M...     -0.096   
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia        -0.002   
                                 ¿Partir de FinBERT supera a entrenar solo con e...       0.26   
                                 ¿Partir de FinBERT supera a preentrenar uno pro...      0.262   

                                                                                    IC 95 % inferior  \
los 300                          ¿Aprender de ejemplos supera al diccionario? (M...            -0.17   
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia              -0.069   
                                 ¿Partir de FinBERT supera a entrenar solo con e...             0.18   
                                 ¿Partir de FinBERT supera a preentrenar uno pro...            0.175   
los 271 que tratan de su empresa ¿Aprender de ejemplos supera al diccionario? (M...           -0.186   
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia              -0.076   
                                 ¿Partir de FinBERT supera a entrenar solo con e...             0.19   
                                 ¿Partir de FinBERT supera a preentrenar uno pro...            0.183   

                                                                                    IC 95 % superior  \
los 300                          ¿Aprender de ejemplos supera al diccionario? (M...            0.004   
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia               0.071   
                                 ¿Partir de FinBERT supera a entrenar solo con e...            0.322   
                                 ¿Partir de FinBERT supera a preentrenar uno pro...            0.331   
los 271 que tratan de su empresa ¿Aprender de ejemplos supera al diccionario? (M...            0.003   
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia               0.074   
                                 ¿Partir de FinBERT supera a entrenar solo con e...            0.331   
                                 ¿Partir de FinBERT supera a preentrenar uno pro...            0.328   

                                                                                             ¿concluyente?  
los 300                          ¿Aprender de ejemplos supera al diccionario? (M...  no (el 0 está dentro)  
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia     no (el 0 está dentro)  
                                 ¿Partir de FinBERT supera a entrenar solo con e...                     sí  
                                 ¿Partir de FinBERT supera a preentrenar uno pro...                     sí  
los 271 que tratan de su empresa ¿Aprender de ejemplos supera al diccionario? (M...  no (el 0 está dentro)  
                                 ¿Aporta preentrenar? (M3 − M2), la única limpia     no (el 0 está dentro)  
                                 ¿Partir de FinBERT supera a entrenar solo con e...                     sí  
                                 ¿Partir de FinBERT supera a preentrenar uno pro...                     sí

Dónde acierta y dónde falla cada red con los 300 (filas: la etiqueta del autor; columnas: lo que dice el modelo):

In [16]:
for modelo in ["M2 Transformer", "M4 FinBERT"]:
    print(modelo)
    display(evaluaciones_a_mano[("los 300", modelo)]["matriz_confusion"])

M2 Transformer


predicha,negative,neutral,positive
real,,,
negative,12,39,4
neutral,5,130,38
positive,4,58,10


M4 FinBERT


predicha,negative,neutral,positive
real,,,
negative,27,26,2
neutral,11,144,18
positive,3,34,35


Algunos titulares en los que M4 se equivoca, elegidos al azar con la semilla:

In [17]:
errores_m4 = a_mano[a_mano["M4 FinBERT"] != a_mano["etiqueta"]]
print(f"M4 se equivoca en {len(errores_m4)} de {len(a_mano)} titulares")
with pd.option_context("display.max_colwidth", 110):
    display(errores_m4.sample(min(10, len(errores_m4)), random_state=SEMILLA)[["titular", "etiqueta", "M4 FinBERT", "M2 Transformer"]])

M4 se equivoca en 94 de 300 titulares


,titular,etiqueta,M4 FinBERT,M2 Transformer
127,Palantir Q3 earnings: Did the AI company just have a 'celestial' quarter?,positive,neutral,neutral
58,Why AT&T Stock is Tumbling Today,negative,neutral,neutral
184,"Humana Is Dropping Plans Covering 600,000 Members and Hopes 240,000 Choose Humana Again. The Replacement I...",negative,neutral,neutral
239,Shopify Stock Is Up 15% Since September 2025. Here’s Why It Trades at a Lower Forward P/E,neutral,negative,positive
3,Do You Like Cash? Consider T-Mobile Stock,positive,neutral,neutral
64,"Nike Climbs 5% as Beaten-Down Consumer Names Catch a Bid; Lululemon Rises 4%, On Holding Dips",negative,positive,neutral
121,ATI Physical Therapy Launches Innovative Online Scheduling Platform to Make Accessing Care Faster and Easier,neutral,positive,positive
220,"Texas Blue Wave Could Threaten 27% Of Planned U.S. Data Center Power. Here’s What It Means For CoreWeave, ...",negative,neutral,neutral
27,"ATI: Stronger Business, Tougher Math",neutral,positive,positive
152,"With Shares Down 41% This Year, Could Buying SoundHound AI (SOUN) Stock Today Set You Up for Life?",neutral,negative,neutral


**Conclusión: fuera de PhraseBank, solo M4 sigue leyendo bien el sentimiento. M2 y M3 se hunden, y el diccionario aguanta mejor que ellos.**

| Modelo | Test de PhraseBank | 300 titulares a mano (IC 95 %) | Los 271 que tratan de su empresa |
|---|---|---|---|
| Siempre neutral | 0,248 | 0,244 | 0,231 |
| M1 diccionario | 0,502 | 0,460 (0,40 – 0,52) | 0,466 |
| M2 Transformer | 0,673 | 0,376 (0,31 – 0,44) | 0,370 |
| M3 BERT | 0,586 | 0,375 (0,32 – 0,43) | 0,368 |
| **M4 FinBERT** | **0,813** | **0,626** (0,56 – 0,69) | **0,630** |

- **M4 es el mejor también aquí, y con claridad:** M4 − M2 = +0,25 (0,18 – 0,32) y M4 − M3 = +0,25 (0,18 – 0,33). Pierde 0,19 respecto a PhraseBank, pero acierta 27 de 55 negativos y 35 de 72 positivos. Su error principal sigue siendo llamar neutral a lo que tiene tono (60 de 127).
- **M2 y M3 pierden casi la mitad (de 0,67 y 0,59 a 0,38).** M2 acierta solo 12 de 55 negativos y 10 de 72 positivos, y llama positivos a 38 titulares neutrales. Aprendieron de 2.901 frases de notas de prensa finlandesas de 2004–2010, y los titulares de hoy tienen otra forma (preguntas, cifras, *clickbait*) y otro vocabulario (*AI*, *data center*, *P/E*).
- **El diccionario apenas pierde (de 0,50 a 0,46).** Sus listas no dependen de las frases de entrenamiento. Aquí M2 − M1 = −0,085 (−0,17 a 0,004): ya no se puede decir que aprender de ejemplos supere al diccionario, aunque tampoco lo contrario, porque el 0 queda justo dentro.
- **Preentrenar uno propio sigue sin aportar:** M3 − M2 = −0,001 (−0,07 a 0,07).
- **Quitar los 29 titulares que no tratan de su empresa no cambia nada:** cada modelo varía menos de 0,01.
- **Los errores de M4** (94 de 300) son sobre todo titulares que cuentan un movimiento de la acción con palabras que no son de tono (*Why AT&T Stock is Tumbling Today*, *Nike Climbs 5%*) o preguntas sin tono claro (*Do You Like Cash?*).

**Para el estudio de eventos:** M4 es el instrumento más fiable, pero no perfecto (F1 de 0,63 en titulares). M2 y M3 miden el sentimiento de los titulares casi tan mal como el diccionario, o peor. Si M4 encuentra más relación con la bolsa que los demás, es coherente con que mide mejor; si ninguno la encuentra, no será por usar un modelo malo.

## 5. ¿Cuánto se mueve cada acción frente al mercado?

Para saber si una noticia mueve una acción no basta con su rentabilidad: un día en que sube toda la bolsa, casi todas las acciones suben, haya noticias o no. Se mide la **rentabilidad anormal**: la de la acción menos la del mercado ese mismo día.

$$\text{rentabilidad anormal} = \frac{\text{cierre}_t}{\text{cierre}_{t-1}} - 1 \;-\; \left(\frac{\text{SPY}_t}{\text{SPY}_{t-1}} - 1\right)$$

**El mercado es SPY**, el fondo que replica el S&P 500 (las 500 mayores empresas de EEUU). Es el modelo más sencillo de los estudios de eventos, el *ajustado al mercado* (MacKinlay, 1997): no estima la beta de cada acción. Una acción que suele moverse el doble que el mercado tendrá rentabilidades anormales que en parte son solo mercado, pero eso afecta igual a sus días con noticias buenas y malas, que es lo que se compara.

SPY se descarga de yfinance **con el mismo ajuste por splits y dividendos y las mismas fechas que los precios del notebook 02**, y se guarda en caché.

In [18]:
RUTA_SPY = CARPETA_CACHE / "precios_spy.parquet"
INICIO_PRECIOS, FIN_PRECIOS = "2016-10-10", "2026-10-10"   # las del notebook 02 (el fin no se incluye)

if RUTA_SPY.exists():
    spy = pd.read_parquet(RUTA_SPY)
    print("SPY leído de caché")
else:
    crudo = yf.download("SPY", start=INICIO_PRECIOS, end=FIN_PRECIOS, auto_adjust=True, progress=False)
    if crudo.empty:
        raise RuntimeError("yfinance no devolvió datos de SPY: revisa la conexión o vuelve a intentarlo más tarde")
    print("Así llega de yfinance (columnas con dos niveles: el dato y el ticker):")
    display(crudo.head(3))
    spy = (crudo.xs("SPY", axis=1, level="Ticker").reset_index()
           .rename(columns={"Date": "fecha", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})
           [["fecha", "open", "high", "low", "close", "volume"]])
    spy.to_parquet(RUTA_SPY, index=False)
    print("SPY descargado y guardado")
print(f"SPY: {len(spy):,} sesiones, del {spy['fecha'].min():%d %b %Y} al {spy['fecha'].max():%d %b %Y}")

SPY leído de caché
SPY: 2,514 sesiones, del 10 Oct 2016 al 09 Oct 2026


La rentabilidad de cada día, para cada acción y para SPY, y la diferencia. Cada acción se ordena por fecha y se compara con su cierre anterior (`pct_change` por empresa), así que el primer día de cada una queda vacío y se quita.

In [19]:
precios = (pd.read_parquet(CARPETA_CACHE / "precios_historicos.parquet", columns=["fecha", "ticker", "close"])
           .sort_values(["ticker", "fecha"]).reset_index(drop=True))
precios["rentabilidad"] = precios.groupby("ticker")["close"].pct_change()
spy["rentabilidad_spy"] = spy["close"].pct_change()

rentabilidades = precios.merge(spy[["fecha", "rentabilidad_spy"]], on="fecha", how="left")
rentabilidades["anormal"] = rentabilidades["rentabilidad"] - rentabilidades["rentabilidad_spy"]
rentabilidades = rentabilidades.dropna(subset=["anormal"]).reset_index(drop=True)
print(f"Rentabilidades diarias: {len(rentabilidades):,} de {rentabilidades['ticker'].nunique()} empresas")
rentabilidades.head()

Rentabilidades diarias: 2,280,091 de 1000 empresas


,fecha,ticker,close,rentabilidad,rentabilidad_spy,anormal
0,2016-10-11,A,42.283371,-0.041938,-0.012630,-0.029308
1,2016-10-12,A,42.079792,-0.004815,0.001312,-0.006126
2,2016-10-13,A,41.792892,-0.006818,-0.003275,-0.003543
3,2016-10-14,A,41.718864,-0.001771,0.000516,-0.002288
4,2016-10-17,A,41.783638,0.001553,-0.003472,0.005025


**¿Cómo son en el periodo de las noticias?** De octubre de 2016 a junio de 2020, que es lo que cubre FNSPID. En porcentaje:

In [20]:
PERIODO_NOTICIAS = (pd.Timestamp("2016-10-01"), pd.Timestamp("2020-07-10"))   # FNSPID llega al 11 jun 2020, y algo de margen
en_periodo = rentabilidades[rentabilidades["fecha"].between(*PERIODO_NOTICIAS)]

resumen = (en_periodo[["rentabilidad", "rentabilidad_spy", "anormal"]] * 100).describe(percentiles=[0.01, 0.5, 0.99]).T
display(resumen[["count", "mean", "std", "min", "1%", "50%", "99%", "max"]].round(3))
for umbral in [0.10, 0.25, 0.50]:
    print(f"Días con una rentabilidad anormal de más del ±{umbral:.0%}: {(en_periodo['anormal'].abs() > umbral).sum():,}")
print("Los más extremos:")
en_periodo.loc[en_periodo["anormal"].abs().nlargest(5).index, ["fecha", "ticker", "rentabilidad", "rentabilidad_spy", "anormal"]].round(3)

,count,mean,std,min,1%,50%,99%,max
rentabilidad,784896.0,0.075,2.742,-72.040,-7.442,0.079,7.795,386.667
rentabilidad_spy,784896.0,0.057,1.276,-10.942,-4.182,0.066,3.357,9.060
anormal,784896.0,0.019,2.375,-66.958,-6.112,-0.001,6.630,386.560


Días con una rentabilidad anormal de más del ±10%: 5,198
Días con una rentabilidad anormal de más del ±25%: 402
Días con una rentabilidad anormal de más del ±50%: 41
Los más extremos:


,fecha,ticker,rentabilidad,rentabilidad_spy,anormal
469397,2018-09-19,CLSK,3.867,0.001,3.866
1275873,2017-11-24,MARA,1.729,0.002,1.727
440994,2020-03-13,CHRD,1.703,0.085,1.617
232554,2019-01-07,AXSM,1.612,0.008,1.604
1293333,2018-05-31,MDGL,1.450,-0.006,1.456


SPY en todo el histórico, con el periodo de las noticias sombreado. La caída de marzo de 2020 (la COVID) queda dentro: son los días de mayor volatilidad de todo el periodo.

In [21]:
grafico = px.line(spy, x="fecha", y="close", title="SPY (cierre ajustado) y periodo de las noticias de FNSPID",
                  labels={"close": "cierre ajustado (USD)", "fecha": ""})
grafico.add_vrect(x0=PERIODO_NOTICIAS[0], x1=PERIODO_NOTICIAS[1], opacity=0.15, line_width=0, annotation_text="noticias de FNSPID")
grafico.show(renderer="notebook_connected")

**Conclusión:** SPY tiene exactamente las mismas 2.514 sesiones que las acciones, así que cada rentabilidad tiene su referencia del mercado: 2,28 millones de rentabilidades anormales, 784.896 en el periodo de las noticias. La mediana es prácticamente 0 y el 98 % de los días queda entre −6,1 % y +6,6 %. Pero hay **colas extremas**: 41 días por encima de ±50 %, con un máximo de +387 % (CLSK, en 2018). Son sobre todo empresas que en 2016–2020 eran pequeñas y hoy están entre las 1.000 elegidas por su tamaño de 2026 (el sesgo de selección). Unos pocos días así pueden mover la media de un grupo entero, y habrá que tenerlo en cuenta en el paso 6.

## 6. ¿Se mueve la acción después de una noticia? (estudio de eventos)

La pregunta central del proyecto: después de los días con noticias positivas de una empresa, ¿le va mejor a su acción que después de los días con noticias negativas? Se mide con los cuatro modelos. El diseño se fijó antes de ver ningún resultado:

- **Un evento es una empresa en una sesión.** Cada noticia se asigna a la **última sesión que no es posterior a ella**: la del mismo día si es de lunes a viernes, y la del viernes si es de fin de semana. Las noticias del viernes, el sábado y el domingo de una empresa forman un solo evento, porque comparten la misma rentabilidad posterior; si se contaran por separado, esa rentabilidad entraría varias veces.
- **El sentimiento del evento** es la media del sentimiento (−1, 0 o 1) de sus titulares, y su signo decide el grupo: **positivo, neutral o negativo**.
- **Sin mirar al futuro:** FNSPID solo trae el día de publicación, no la hora. Así que la rentabilidad que cuenta es la **de la sesión siguiente**: del cierre de la sesión de la noticia al cierre siguiente, menos SPY.
- **La del mismo día se muestra aparte** y está **inflada**: muchos titulares cuentan un movimiento que ya ha ocurrido (*X shares fall after earnings*). Solo para eventos con alguna noticia en día de sesión, porque en los de fin de semana el movimiento del viernes es anterior a la noticia.
- **Colas recortadas (*winsorización*):** cada rentabilidad anormal se limita a los percentiles 1 y 99 de todas las del periodo, para que ningún día de +300 % decida el resultado. Es lo habitual en finanzas. Como sensibilidad se repite sin recortar.
- **La diferencia entre positivos y negativos** lleva un intervalo de confianza de **Welch** (`diferencia_medias_welch`, en `funciones.py`). **Limitación:** Welch supone eventos independientes, pero los del mismo día se mueven juntos (un día de pánico afecta a todos), así que el intervalo es algo más estrecho de lo que debería. Una diferencia justo en el límite no se presenta como hallazgo.

### 6.1 De titulares a eventos

Cada titular se asigna a su sesión de referencia y a la siguiente. Las noticias anteriores al primer precio (10 oct 2016) no tienen sesión y quedan fuera.

In [22]:
SESIONES = pd.DatetimeIndex(np.sort(spy["fecha"].unique()))
posicion = SESIONES.searchsorted(sentimiento["fecha"], side="right") - 1   # la última sesión que no es posterior a la noticia
con_sesion = posicion >= 0
print(f"Titulares anteriores al primer precio, fuera: {(~con_sesion).sum() // len(MODELOS)} por modelo")

titulares_con_sesion = sentimiento[con_sesion].assign(
    sesion=SESIONES[posicion[con_sesion]],
    siguiente=SESIONES[posicion[con_sesion] + 1],
    en_dia_de_sesion=lambda tabla: tabla["fecha"].isin(SESIONES),
)
titulares_con_sesion[["ticker", "fecha", "sesion", "siguiente", "en_dia_de_sesion", "titular"]].drop_duplicates("titular").sample(5, random_state=SEMILLA)

Titulares anteriores al primer precio, fuera: 722 por modelo


,ticker,fecha,sesion,siguiente,en_dia_de_sesion,titular
30442,BUD,2018-09-21,2018-09-21,2018-09-24,True,EU looking to fine Anheuser-Busch InBev
50527,CTSH,2018-02-08,2018-02-08,2018-02-09,True,Morgan Stanley Maintains Equal-Weight on Cogni...
182778,VTR,2019-06-20,2019-06-20,2019-06-21,True,"Citigroup Upgrades Ventas to Buy, Raises Price..."
143762,PFE,2020-04-22,2020-04-22,2020-04-23,True,Pfizer shares are trading higher after the com...
74618,FLEX,2018-04-26,2018-04-26,2018-04-27,True,"Flex Sees Q1 EPS $0.22-$0.26 vs $0.31 Est., Sa..."


Los titulares de cada empresa y sesión se juntan en un evento, con la media de su sentimiento y su grupo. Después se le añade la rentabilidad anormal de la sesión siguiente y la del mismo día.

In [23]:
GRUPO_POR_SIGNO = {1: "positivo", 0: "neutral", -1: "negativo"}

eventos = (titulares_con_sesion.groupby(["modelo", "ticker", "sesion", "siguiente"])
           .agg(sentimiento_medio=("sentimiento", "mean"), titulares=("titular", "size"),
                alguno_en_dia_de_sesion=("en_dia_de_sesion", "any"))
           .reset_index())
eventos["grupo"] = np.sign(eventos["sentimiento_medio"]).astype(int).map(GRUPO_POR_SIGNO)

anormales = rentabilidades[["ticker", "fecha", "anormal"]]
eventos = (eventos
           .merge(anormales.rename(columns={"fecha": "siguiente", "anormal": "anormal_siguiente"}), on=["ticker", "siguiente"], how="left")
           .merge(anormales.rename(columns={"fecha": "sesion", "anormal": "anormal_mismo_dia"}), on=["ticker", "sesion"], how="left"))
eventos.loc[~eventos["alguno_en_dia_de_sesion"], "anormal_mismo_dia"] = np.nan   # solo de fin de semana: el viernes es anterior
sin_rentabilidad = eventos["anormal_siguiente"].isna()
print(f"Eventos sin rentabilidad en la sesión siguiente (la empresa no cotizaba): {sin_rentabilidad.sum() // len(MODELOS)} por modelo")
eventos = eventos[~sin_rentabilidad].reset_index(drop=True)

display(eventos.groupby(["modelo", "grupo"]).size().unstack()[["positivo", "neutral", "negativo"]])
print(f"Eventos por modelo: {len(eventos) // len(MODELOS):,} · titulares por evento: media {eventos['titulares'].mean():.2f}")

Eventos sin rentabilidad en la sesión siguiente (la empresa no cotizaba): 696 por modelo


grupo,positivo,neutral,negativo
modelo,,,
M1 diccionario,17469,75812,17732
M2 Transformer,35171,66446,9396
M3 BERT,52426,53716,4871
M4 FinBERT,32910,62588,15515


Eventos por modelo: 111,013 · titulares por evento: media 1.71


**Las colas, recortadas.** Los límites son los percentiles 1 y 99 de todas las rentabilidades anormales del periodo de las noticias (sección 5), no solo de los eventos, y se fijan antes de mirar ningún grupo.

In [24]:
LIMITES = en_periodo["anormal"].quantile([0.01, 0.99]).to_numpy()
print(f"Límites: del {LIMITES[0]:.2%} al {LIMITES[1]:.2%}")
for columna in ["anormal_siguiente", "anormal_mismo_dia"]:
    eventos[f"{columna}_recortada"] = eventos[columna].clip(*LIMITES)
recortados = (eventos["anormal_siguiente"] != eventos["anormal_siguiente_recortada"]).mean()
print(f"Eventos con la rentabilidad de la sesión siguiente recortada: {recortados:.1%}")

Límites: del -6.11% al 6.63%
Eventos con la rentabilidad de la sesión siguiente recortada: 2.9%


### 6.2 El resultado: rentabilidad anormal de la sesión siguiente

Para cada modelo: cuántos eventos hay en cada grupo, la rentabilidad anormal media de cada uno (en %) y la diferencia entre positivos y negativos con su intervalo del 95 %.

In [25]:
def estudio_de_eventos(columna: str) -> pd.DataFrame:
    """Para cada modelo, la media de `columna` (en %) por grupo y la diferencia positivos − negativos con su intervalo de Welch."""
    plural = {"positivo": "positivos", "neutral": "neutrales", "negativo": "negativos"}
    filas = {}
    for modelo in MODELOS:
        del_modelo = eventos[(eventos["modelo"] == modelo) & eventos[columna].notna()]
        por_grupo = {grupo: del_modelo.loc[del_modelo["grupo"] == grupo, columna] for grupo in ["positivo", "neutral", "negativo"]}
        welch = diferencia_medias_welch(por_grupo["positivo"], por_grupo["negativo"])
        filas[modelo] = {
            **{f"eventos {plural[grupo]}": len(valores) for grupo, valores in por_grupo.items()},
            **{f"media {plural[grupo]} (%)": 100 * valores.mean() for grupo, valores in por_grupo.items()},
            "positivos − negativos (%)": 100 * welch["diferencia"],
            "IC 95 % inferior": 100 * welch["ic_inferior"], "IC 95 % superior": 100 * welch["ic_superior"],
            "p-valor": welch["p_valor"],
        }
    return pd.DataFrame(filas).T.astype({f"eventos {nombre}": int for nombre in plural.values()}).round(4)


resultado_siguiente = estudio_de_eventos("anormal_siguiente_recortada")
resultado_siguiente

,eventos positivos,eventos neutrales,eventos negativos,media positivos (%),media neutrales (%),media negativos (%),positivos − negativos (%),IC 95 % inferior,IC 95 % superior,p-valor
M1 diccionario,17469,75812,17732,0.0660,0.0497,-0.0774,0.1434,0.0995,0.1873,0.0000
M2 Transformer,35171,66446,9396,0.0363,0.0275,0.0479,-0.0117,-0.0678,0.0445,0.6842
M3 BERT,52426,53716,4871,0.0463,0.0199,0.0105,0.0358,-0.0350,0.1067,0.3214
M4 FinBERT,32910,62588,15515,0.1253,0.0126,-0.0876,0.2129,0.1682,0.2576,0.0000


**Sensibilidad:** lo mismo sin recortar las colas.

In [26]:
estudio_de_eventos("anormal_siguiente")

,eventos positivos,eventos neutrales,eventos negativos,media positivos (%),media neutrales (%),media negativos (%),positivos − negativos (%),IC 95 % inferior,IC 95 % superior,p-valor
M1 diccionario,17469,75812,17732,0.0906,0.0667,-0.1297,0.2203,0.1603,0.2804,0.0000
M2 Transformer,35171,66446,9396,0.0496,0.0334,0.0405,0.0091,-0.0698,0.0880,0.8209
M3 BERT,52426,53716,4871,0.0516,0.0288,0.0183,0.0333,-0.0716,0.1382,0.5336
M4 FinBERT,32910,62588,15515,0.1751,0.0138,-0.1473,0.3224,0.2599,0.3849,0.0000


### 6.3 Aparte: la rentabilidad del mismo día (relación inflada)

La sesión en la que se publica la noticia. Aquí se espera una diferencia grande, pero no indica que la noticia anticipe nada: muchos titulares cuentan un movimiento que ya ha ocurrido.

In [27]:
resultado_mismo_dia = estudio_de_eventos("anormal_mismo_dia_recortada")
resultado_mismo_dia

,eventos positivos,eventos neutrales,eventos negativos,media positivos (%),media neutrales (%),media negativos (%),positivos − negativos (%),IC 95 % inferior,IC 95 % superior,p-valor
M1 diccionario,17097,73714,17473,0.4389,0.1113,-0.4955,0.9344,0.8832,0.9855,0.0
M2 Transformer,34546,64703,9035,0.2012,0.0270,-0.1828,0.3840,0.3247,0.4434,0.0
M3 BERT,51545,52060,4679,0.1480,0.0097,-0.2312,0.3791,0.3024,0.4559,0.0
M4 FinBERT,32358,60824,15102,0.5513,0.0171,-0.7832,1.3345,1.2841,1.3849,0.0


En un gráfico: la rentabilidad anormal media de cada grupo, con su intervalo del 95 %, el mismo día y en la sesión siguiente (colas recortadas).

In [28]:
def media_con_intervalo(valores: pd.Series) -> pd.Series:
    """Media en % y la mitad del ancho de su intervalo del 95 % (t de Student)."""
    mitad = stats.t.ppf(0.975, len(valores) - 1) * valores.std(ddof=1) / np.sqrt(len(valores))
    return pd.Series({"media (%)": 100 * valores.mean(), "error (%)": 100 * mitad})


medias = pd.concat({
    momento: eventos.dropna(subset=[columna]).groupby(["modelo", "grupo"])[columna].apply(media_con_intervalo).unstack()
    for momento, columna in [("el mismo día", "anormal_mismo_dia_recortada"), ("la sesión siguiente", "anormal_siguiente_recortada")]
}, names=["momento"]).reset_index()
grafico = px.bar(medias, x="modelo", y="media (%)", error_y="error (%)", color="grupo", barmode="group", facet_col="momento",
                 category_orders={"grupo": ["negativo", "neutral", "positivo"], "momento": ["el mismo día", "la sesión siguiente"]},
                 color_discrete_map={"negativo": "#d62728", "neutral": "#7f7f7f", "positivo": "#2ca02c"},
                 title="Rentabilidad anormal media según el sentimiento de las noticias (IC 95 %)")
grafico.update_yaxes(matches=None)
grafico.show(renderer="notebook_connected")

**Conclusión: sí hay relación, pero solo la encuentran los modelos que leen bien los titulares, y es pequeña.**

| Modelo | F1 con los 300 titulares a mano | Positivos − negativos, sesión siguiente (IC 95 %) | El mismo día |
|---|---|---|---|
| M1 diccionario | 0,460 | **+0,14 %** (0,10 – 0,19) | +0,93 % |
| M2 Transformer | 0,376 | −0,01 % (−0,07 – 0,04) | +0,38 % |
| M3 BERT | 0,375 | +0,04 % (−0,04 – 0,11) | +0,38 % |
| **M4 FinBERT** | **0,626** | **+0,21 %** (0,17 – 0,26) | +1,33 % |

- **En la sesión siguiente**, con M4, las acciones con noticias positivas le sacan al mercado un 0,13 % y las de negativas pierden un 0,09 %: una diferencia de **0,21 puntos**. Con M1 es de 0,14. Con M2 y M3, los que peor leen titulares, no hay diferencia: el 0 queda dentro. **El modelo que mejor mide el sentimiento es el que más relación encuentra.**
- **Sin recortar las colas** el orden y el signo se mantienen (M4 +0,32 %, M1 +0,22 %; M2 y M3 sin diferencia).
- **El mismo día**, la diferencia es entre 4 y 6 veces mayor (M4: +1,33 %), pero está inflada: muchos titulares cuentan un movimiento que ya ha ocurrido.
- **Cuidado al leerlo:** sin la hora de publicación, una noticia publicada después del cierre se refleja en la sesión siguiente. Así que esto no demuestra que se pueda anticipar la bolsa con las noticias. Además, el intervalo de Welch supone eventos independientes, y es algo optimista.

## Comprobaciones

In [29]:
# Plantilla: 300 titulares distintos, ids del 1 al 300, como mucho 5 por empresa, y la misma muestra que se genera aquí
assert len(etiquetado) == TITULARES_A_MANO and etiquetado["id"].tolist() == list(range(1, TITULARES_A_MANO + 1))
assert etiquetado["titular"].is_unique
assert etiquetado["ticker"].value_counts().max() <= MAXIMO_POR_EMPRESA
assert etiquetado[["id", "ticker", "titular"]].equals(plantilla[["id", "ticker", "titular"]])
assert len(mal_escritos) == 0 and len(no_neutrales_ajenos) == 0
print("Plantilla: OK")

Plantilla: OK


In [30]:
# Modelos: los tres repiten exactamente lo que dijeron en el 07 sobre la validación de PhraseBank
assert keras.backend.backend() == "tensorflow"
assert all(proporcion == 1 for proporcion in coinciden_con_el_07.values()), coinciden_con_el_07
print("Modelos: OK")

Modelos: OK


In [31]:
# Sentimiento: una fila por titular y modelo, los mismos titulares que en FNSPID, valores válidos y sin huecos
assert len(sentimiento) == len(MODELOS) * len(titulares) == 4 * 192069
assert (sentimiento["modelo"].value_counts() == len(titulares)).all()
assert not sentimiento.duplicated(["ticker", "fecha", "titular", "modelo"]).any()
for nombre in MODELOS:
    del_modelo = sentimiento[sentimiento["modelo"] == nombre].sort_values(["ticker", "fecha", "titular"])
    assert del_modelo[["ticker", "fecha", "titular"]].reset_index(drop=True).equals(
        titulares.sort_values(["ticker", "fecha", "titular"]).reset_index(drop=True))
assert sentimiento["sentimiento"].isin([-1, 0, 1]).all() and sentimiento.notna().all().all()
assert sentimiento["confianza"].between(0, 1).all()
assert (sentimiento.loc[sentimiento["modelo"] != "M1 diccionario", "confianza"] >= 1 / 3 - 1e-6).all()
# M1 con confianza 0 (ninguna palabra del diccionario) es siempre neutral
m1_sin_palabras = sentimiento[(sentimiento["modelo"] == "M1 diccionario") & (sentimiento["confianza"] == 0)]
assert (m1_sin_palabras["sentimiento"] == 0).all()
print("Sentimiento de los titulares: OK")

Sentimiento de los titulares: OK


In [32]:
# Titulares a mano: los 300 puntuados por los cuatro modelos con clases válidas, 271 que tratan de su empresa,
# y cada diferencia dentro de su intervalo
assert len(a_mano) == 300 and (a_mano["trata_de_la_empresa"] == "sí").sum() == 271
assert all(a_mano[modelo].isin(CLASES_SENTIMIENTO).all() for modelo in MODELOS_A_MANO)
assert all(e["frases"] == len(CONJUNTOS_A_MANO[conjunto]) for (conjunto, _), e in evaluaciones_a_mano.items())
assert all(d["IC 95 % inferior"] <= d["diferencia"] <= d["IC 95 % superior"] for d in diferencias_a_mano.values())
print("Titulares a mano: OK")

Titulares a mano: OK


In [33]:
# SPY y rentabilidades: las mismas sesiones que las acciones, sin huecos, y la anormal es la resta de las dos
assert spy["fecha"].is_unique and spy[["fecha", "close"]].notna().all().all() and (spy["close"] > 0).all()
assert set(precios["fecha"]) == set(spy["fecha"])                        # cada sesión de las acciones tiene su SPY
assert not rentabilidades.duplicated(["ticker", "fecha"]).any()
assert rentabilidades[["rentabilidad", "rentabilidad_spy", "anormal"]].notna().all().all()
assert np.allclose(rentabilidades["anormal"], rentabilidades["rentabilidad"] - rentabilidades["rentabilidad_spy"])
assert len(rentabilidades) == len(precios) - precios["ticker"].nunique()   # solo falta el primer día de cada empresa
print("SPY y rentabilidades: OK")

SPY y rentabilidades: OK


In [34]:
# Estudio de eventos: sin mirar al futuro, un evento por modelo, empresa y sesión, el grupo según el signo,
# las colas recortadas a los límites fijados y la función de Welch igual que la fórmula
assert (titulares_con_sesion["sesion"] <= titulares_con_sesion["fecha"]).all()        # la sesión de la noticia no es posterior
assert (titulares_con_sesion["siguiente"] > titulares_con_sesion["fecha"]).all()      # la rentabilidad que cuenta es posterior
assert not eventos.duplicated(["modelo", "ticker", "sesion"]).any()
assert (eventos.groupby("modelo").size().nunique() == 1)                              # los cuatro modelos, los mismos eventos
assert (np.sign(eventos["sentimiento_medio"]).astype(int).map(GRUPO_POR_SIGNO) == eventos["grupo"]).all()
assert eventos["anormal_siguiente_recortada"].between(*LIMITES).all()
assert eventos.loc[~eventos["alguno_en_dia_de_sesion"], "anormal_mismo_dia"].isna().all()
a, b = eventos["anormal_siguiente"].iloc[:500].to_numpy(), eventos["anormal_siguiente"].iloc[500:1300].to_numpy()
welch = diferencia_medias_welch(a, b)
error = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
assert abs(welch["diferencia"] - (a.mean() - b.mean())) < 1e-12 and welch["ic_inferior"] < welch["diferencia"] < welch["ic_superior"]
assert abs((welch["ic_superior"] - welch["ic_inferior"]) / 2 - 1.96 * error) / error < 0.01   # con tantos datos, t ≈ normal
print("Estudio de eventos: OK")

Estudio de eventos: OK
